# Classification Algorithms

Master classification notebook for the Machine Learning Algorithms Laboratory. Experiment 2 begins with Naive Bayes and KNN.

## Aim and Objective

Build and compare classification models using Naive Bayes and KNN with timing, tuning, cross-validation, plots, and analysis.

## Theory

Naive Bayes is a probabilistic classifier based on Bayes theorem. KNN is a lazy learning method that classifies samples using nearby training points.

## Import Libraries

Import model, metric, tuning, timing, and plotting libraries for classification.

In [4]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve, auc
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelBinarizer, MinMaxScaler

sns.set_theme(style="whitegrid")

## Configuration

Change only this cell for future datasets. The EDA notebook uses these values and returns preprocessed outputs.

In [5]:
DATASET_NAME = "Spambase"
CSV_PATH = "spambase.csv"
TARGET_COLUMN = "class"
TEST_SIZE = 0.2
RANDOM_STATE = 42

EDA_NOTEBOOK_PATH = "../Ass1/EDA_Pipeline.ipynb"
OUTPUT_DIRECTORY = "../Ass1/eda_output"
CLASSIFICATION_OUTPUT = "classification_output"
CLASSIFICATION_PLOTS_DIRECTORY = os.path.join(CLASSIFICATION_OUTPUT, "plots")
os.makedirs(CLASSIFICATION_PLOTS_DIRECTORY, exist_ok=True)

## Run EDA Notebook

Execute Assignment 1 and reuse its processed train-test split, feature names, preprocessor, and detected problem type.

In [6]:
%run $EDA_NOTEBOOK_PATH

assert problem_type == "classification", "This notebook requires a classification target."
print("Received preprocessed data from EDA notebook")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"Features: {len(feature_names)}")
display(pd.Series(y_train).value_counts().rename_axis(TARGET_COLUMN).reset_index(name="train_count"))

Exception: File `'$Desktop/ML/Ass1/EDA_Pipeline.ipynb'` not found.

## Reusable Evaluation Pipeline

All current and future classifiers use the same training, timing, metric, and plotting helpers.

In [ ]:
results = []
model_store = {}
classes = np.array(sorted(pd.Series(y_train).dropna().unique()))

def predict_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return scores if scores.ndim > 1 else np.vstack([1 - scores, scores]).T
    return None

def metric_dict(y_true, y_pred, y_score=None):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "ROC-AUC": np.nan,
    }
    if y_score is not None:
        try:
            metrics["ROC-AUC"] = roc_auc_score(y_true, y_score, multi_class="ovr", average="weighted", labels=classes) if len(classes) > 2 else roc_auc_score(y_true, y_score[:, 1])
        except Exception:
            pass
    return metrics

def plot_confusion_matrix(model_name, y_true, y_pred):
    plt.figure(figsize=(5, 4))
    sns.heatmap(confusion_matrix(y_true, y_pred, labels=classes), annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(CLASSIFICATION_PLOTS_DIRECTORY, f"confusion_matrix_{model_name}.png"), bbox_inches="tight")
    plt.show()

def plot_roc_pr(model_name, y_true, y_score):
    if y_score is None:
        return
    lb = LabelBinarizer().fit(classes)
    y_bin = lb.transform(y_true)
    y_bin = np.column_stack([1 - y_bin, y_bin]) if len(classes) == 2 and y_bin.shape[1] == 1 else y_bin
    plt.figure(figsize=(6, 4))
    for i, label in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        plt.plot(fpr, tpr, label=f"{label} AUC={auc(fpr, tpr):.3f}")
    plt.plot([0, 1], [0, 1], "k--")
    plt.title(f"ROC Curve - {model_name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(CLASSIFICATION_PLOTS_DIRECTORY, f"roc_curve_{model_name}.png"), bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(6, 4))
    for i, label in enumerate(classes):
        precision, recall, _ = precision_recall_curve(y_bin[:, i], y_score[:, i])
        plt.plot(recall, precision, label=str(label))
    plt.title(f"Precision-Recall Curve - {model_name}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(CLASSIFICATION_PLOTS_DIRECTORY, f"precision_recall_{model_name}.png"), bbox_inches="tight")
    plt.show()

def train_and_evaluate_model(model_name, estimator, Xtr=X_train, Xte=X_test, ytr=y_train, yte=y_test, plot=True):
    model = clone(estimator)
    start = time.perf_counter()
    model.fit(Xtr, ytr)
    training_time = time.perf_counter() - start
    start = time.perf_counter()
    y_pred = model.predict(Xte)
    prediction_time = time.perf_counter() - start
    y_score = predict_scores(model, Xte)
    row = {"Model": model_name, **metric_dict(yte, y_pred, y_score), "Training Time": training_time, "Prediction Time": prediction_time}
    results.append(row)
    model_store[model_name] = model
    display(pd.DataFrame([row]))
    print(classification_report(yte, y_pred, zero_division=0))
    if plot:
        plot_confusion_matrix(model_name, yte, y_pred)
        plot_roc_pr(model_name, yte, y_score)
    return model, row

## Gaussian Naive Bayes

Train and evaluate Gaussian Naive Bayes using the common evaluation pipeline.

In [ ]:
gaussian_model, gaussian_row = train_and_evaluate_model("Gaussian NB", GaussianNB())

## Multinomial Naive Bayes

Use a model-specific non-negative transform because Multinomial NB requires non-negative inputs.

In [ ]:
multinomial_model, multinomial_row = train_and_evaluate_model("Multinomial NB", Pipeline([("minmax", MinMaxScaler()), ("model", MultinomialNB())]))

## Bernoulli Naive Bayes

Train and evaluate Bernoulli Naive Bayes using the common evaluation pipeline.

In [ ]:
bernoulli_model, bernoulli_row = train_and_evaluate_model("Bernoulli NB", BernoulliNB())

## KNN: Effect of k

Evaluate KNN for k values specified in Experiment 2 and plot accuracy versus k.

In [ ]:
knn_rows = []
for k in [1, 3, 5, 7, 9, 11]:
    model, row = train_and_evaluate_model(f"KNN k={k}", KNeighborsClassifier(n_neighbors=k), plot=False)
    knn_rows.append({"k": k, **{m: row[m] for m in ["Accuracy", "Precision", "Recall", "F1", "Training Time", "Prediction Time"]}})

knn_comparison = pd.DataFrame(knn_rows)
display(knn_comparison)

plt.figure(figsize=(7, 4))
sns.lineplot(data=knn_comparison, x="k", y="Accuracy", marker="o")
plt.title("KNN Accuracy vs k")
plt.tight_layout()
plt.savefig(os.path.join(CLASSIFICATION_PLOTS_DIRECTORY, "accuracy_vs_k.png"), bbox_inches="tight")
plt.show()

best_k = int(knn_comparison.sort_values(["F1", "Accuracy"], ascending=False).iloc[0]["k"])
best_knn_model = model_store[f"KNN k={best_k}"]
print(f"Best k based on F1: {best_k}")
plot_confusion_matrix(f"Best KNN k={best_k}", y_test, best_knn_model.predict(X_test))
plot_roc_pr(f"Best KNN k={best_k}", y_test, predict_scores(best_knn_model, X_test))

## Hyperparameter Tuning

Compare GridSearchCV and RandomizedSearchCV for KNN.

In [ ]:
param_grid = {"n_neighbors": [1, 3, 5, 7, 9, 11], "weights": ["uniform", "distance"], "algorithm": ["auto", "brute", "kd_tree", "ball_tree"], "metric": ["euclidean", "manhattan"]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

start = time.perf_counter()
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, scoring="accuracy", cv=cv, n_jobs=-1)
grid_search.fit(X_train, y_train)
grid_time = time.perf_counter() - start

start = time.perf_counter()
random_search = RandomizedSearchCV(KNeighborsClassifier(), param_grid, n_iter=20, scoring="accuracy", cv=cv, random_state=RANDOM_STATE, n_jobs=-1)
random_search.fit(X_train, y_train)
random_time = time.perf_counter() - start

tuning_table = pd.DataFrame({
    "Parameter": ["Best k", "Metric", "Weights", "Algorithm", "CV Accuracy", "Execution Time"],
    "GridSearchCV": [grid_search.best_params_["n_neighbors"], grid_search.best_params_["metric"], grid_search.best_params_["weights"], grid_search.best_params_["algorithm"], grid_search.best_score_, grid_time],
    "RandomizedSearchCV": [random_search.best_params_["n_neighbors"], random_search.best_params_["metric"], random_search.best_params_["weights"], random_search.best_params_["algorithm"], random_search.best_score_, random_time],
})
display(tuning_table)

grid_results = pd.DataFrame(grid_search.cv_results_)
heatmap_data = grid_results.groupby(["param_n_neighbors", "param_weights"])["mean_test_score"].max().unstack()
plt.figure(figsize=(7, 4))
sns.heatmap(heatmap_data, annot=True, cmap="viridis", fmt=".3f")
plt.title("GridSearchCV KNN Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(CLASSIFICATION_PLOTS_DIRECTORY, "grid_search_heatmap.png"), bbox_inches="tight")
plt.show()

random_results = pd.DataFrame(random_search.cv_results_)
plt.figure(figsize=(7, 4))
sns.histplot(random_results["mean_test_score"], bins=10, kde=True)
plt.title("RandomizedSearchCV Score Distribution")
plt.tight_layout()
plt.savefig(os.path.join(CLASSIFICATION_PLOTS_DIRECTORY, "random_search_distribution.png"), bbox_inches="tight")
plt.show()

## KDTree vs BallTree

Compare KNN tree algorithms using the same k and distance metric.

In [ ]:
tree_rows = []
for algorithm in ["kd_tree", "ball_tree"]:
    model = KNeighborsClassifier(n_neighbors=best_k, algorithm=algorithm, metric="euclidean")
    start = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - start
    start = time.perf_counter()
    pred = model.predict(X_test)
    pred_time = time.perf_counter() - start
    tree_rows.append({"Algorithm": algorithm, "Accuracy": accuracy_score(y_test, pred), "Training Time": train_time, "Prediction Time": pred_time})

kd_ball_metrics = pd.DataFrame(tree_rows).set_index("Algorithm")
kd_ball_table = pd.DataFrame({
    "Metric": ["Accuracy", "Training Time", "Prediction Time"],
    "KDTree": [kd_ball_metrics.loc["kd_tree", "Accuracy"], kd_ball_metrics.loc["kd_tree", "Training Time"], kd_ball_metrics.loc["kd_tree", "Prediction Time"]],
    "BallTree": [kd_ball_metrics.loc["ball_tree", "Accuracy"], kd_ball_metrics.loc["ball_tree", "Training Time"], kd_ball_metrics.loc["ball_tree", "Prediction Time"]],
})
display(kd_ball_table)

## Cross Validation

Perform 5-fold cross-validation for the best Naive Bayes model and best KNN model.

In [ ]:
nb_rows = [row for row in results if row["Model"] in ["Gaussian NB", "Multinomial NB", "Bernoulli NB"]]
best_nb_name = max(nb_rows, key=lambda r: r["F1"])["Model"]
best_nb_estimator = model_store[best_nb_name]
best_knn_estimator = grid_search.best_estimator_

nb_scores = cross_val_score(best_nb_estimator, X_train, y_train, cv=cv, scoring="accuracy")
knn_scores = cross_val_score(best_knn_estimator, X_train, y_train, cv=cv, scoring="accuracy")
cv_table = pd.DataFrame({"Fold": [1, 2, 3, 4, 5], "Naive Bayes": nb_scores, "Best KNN": knn_scores})
cv_table = pd.concat([cv_table, pd.DataFrame([{"Fold": "Average", "Naive Bayes": nb_scores.mean(), "Best KNN": knn_scores.mean()}])], ignore_index=True)
display(cv_table)

cv_plot = cv_table[cv_table["Fold"] != "Average"].melt(id_vars="Fold", var_name="Model", value_name="Accuracy")
plt.figure(figsize=(7, 4))
sns.lineplot(data=cv_plot, x="Fold", y="Accuracy", hue="Model", marker="o")
plt.title("5-Fold Cross-Validation Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(CLASSIFICATION_PLOTS_DIRECTORY, "cross_validation_accuracy.png"), bbox_inches="tight")
plt.show()

## Comparison Tables and Timing Analysis

Generate all required comparison tables from stored model results.

In [ ]:
results_df = pd.DataFrame(results)
nb_comparison = results_df[results_df["Model"].isin(["Gaussian NB", "Multinomial NB", "Bernoulli NB"])].set_index("Model")[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].T
nb_comparison.columns = [col.replace(" NB", "") for col in nb_comparison.columns]
knn_pdf_table = knn_comparison[["k", "Accuracy", "Precision", "Recall", "F1"]]
time_table = results_df[results_df["Model"].isin(["Gaussian NB", "Multinomial NB", "Bernoulli NB", f"KNN k={best_k}"])][["Model", "Training Time", "Prediction Time"]]
complexity_table = pd.DataFrame({
    "Algorithm": ["Naive Bayes", "KNN (Brute)", "KDTree", "BallTree"],
    "Training": ["O(nd)", "O(1)", "O(nlog n)", "O(nlog n)"],
    "Prediction": ["O(d)", "O(nd)", "O(log n) avg", "O(log n) avg"],
})

display(Markdown("### Naive Bayes Comparison")); display(nb_comparison)
display(Markdown("### KNN Comparison")); display(knn_pdf_table)
display(Markdown("### Grid Search vs Randomized Search")); display(tuning_table)
display(Markdown("### KDTree vs BallTree")); display(kd_ball_table)
display(Markdown("### Cross Validation")); display(cv_table)
display(Markdown("### Experimental Time Analysis")); display(time_table)
display(Markdown("### Theoretical Complexity")); display(complexity_table)

plt.figure(figsize=(8, 4))
sns.barplot(data=time_table, x="Model", y="Training Time")
plt.title("Training Time Comparison")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(CLASSIFICATION_PLOTS_DIRECTORY, "training_time_comparison.png"), bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=time_table, x="Model", y="Prediction Time")
plt.title("Prediction Time Comparison")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(CLASSIFICATION_PLOTS_DIRECTORY, "prediction_time_comparison.png"), bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 4))
sns.barplot(data=results_df, x="Model", y="F1")
plt.title("Classifier Comparison by F1 Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(CLASSIFICATION_PLOTS_DIRECTORY, "classifier_comparison.png"), bbox_inches="tight")
plt.show()

## Analysis Questions

Answer the required questions using the computed experimental results.

In [ ]:
best_nb_metric = results_df[results_df["Model"].isin(["Gaussian NB", "Multinomial NB", "Bernoulli NB"])].sort_values("F1", ascending=False).iloc[0]
best_knn_metric = knn_comparison.sort_values(["F1", "Accuracy"], ascending=False).iloc[0]
preferred_search = "GridSearchCV" if grid_search.best_score_ >= random_search.best_score_ else "RandomizedSearchCV"
faster_tree = "KDTree" if kd_ball_table.loc[kd_ball_table["Metric"] == "Prediction Time", "KDTree"].iloc[0] <= kd_ball_table.loc[kd_ball_table["Metric"] == "Prediction Time", "BallTree"].iloc[0] else "BallTree"
large_dataset_choice = best_nb_metric["Model"] if best_nb_metric["F1"] >= best_knn_metric["F1"] * 0.98 else f"KNN k={int(best_knn_metric['k'])}"

analysis_answers = [
    f"Best Naive Bayes variant: {best_nb_metric['Model']} with F1={best_nb_metric['F1']:.4f}.",
    f"Optimal k: {int(best_knn_metric['k'])} with F1={best_knn_metric['F1']:.4f} and Accuracy={best_knn_metric['Accuracy']:.4f}.",
    f"{preferred_search} performed better by CV accuracy; Grid={grid_search.best_score_:.4f}, Randomized={random_search.best_score_:.4f}.",
    f"{faster_tree} had faster prediction time in the KDTree vs BallTree comparison.",
    "Practical timing follows the theory: Naive Bayes trains and predicts quickly, while KNN has low training cost but prediction depends on neighbor search.",
    f"Preferred large-dataset classifier from these results: {large_dataset_choice}.",
]
for answer in analysis_answers:
    print(answer)

## Conclusion and Report

Generate a concise conclusion and save the classification report without rerunning models.

In [ ]:
conclusion = f"For {DATASET_NAME}, {best_nb_metric['Model']} was the best Naive Bayes model, while KNN performed best at k={int(best_knn_metric['k'])}. The final choice should balance F1 score, prediction time, and dataset size."
print(conclusion)

os.makedirs(CLASSIFICATION_OUTPUT, exist_ok=True)
report_path = os.path.join(CLASSIFICATION_OUTPUT, "classification_report.md")
report_parts = [
    "# Classification Report\n",
    f"## Aim\nClassify {DATASET_NAME} samples using Naive Bayes and KNN.\n",
    "## Naive Bayes Comparison\n" + nb_comparison.to_markdown() + "\n",
    "## KNN Comparison\n" + knn_pdf_table.to_markdown(index=False) + "\n",
    "## Grid Search vs Randomized Search\n" + tuning_table.to_markdown(index=False) + "\n",
    "## KDTree vs BallTree\n" + kd_ball_table.to_markdown(index=False) + "\n",
    "## Cross Validation\n" + cv_table.to_markdown(index=False) + "\n",
    "## Experimental Time Analysis\n" + time_table.to_markdown(index=False) + "\n",
    "## Theoretical Complexity\n" + complexity_table.to_markdown(index=False) + "\n",
    "## Analysis\n" + "\n".join(f"- {answer}" for answer in analysis_answers) + "\n",
    "## Conclusion\n" + conclusion + "\n",
]
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_parts))

print(f"Report saved to: {report_path}")
print(f"Plots saved to: {CLASSIFICATION_PLOTS_DIRECTORY}")